---
title: Methods for uncertainty quantification
---

## Bayesian inference

The gold standard for inference with error propagation is Bayesian inference which is a from-first-principles approach. Assuming the probability distribution from which the input data are taken is known, one can, using Bayes' theorem, calculate the probability distribution for the output variable. Hence one would know not only the most probable value, but also the uncertainty of this value. 

This takes us back to the fundamental inference problem we really want to solve using an ML model or some other technique. Given a set of possible classes^[For simplicity I limit myself to the case of classification here, but a similar procedure can be done for regression problems] and some measurement data, our task is to determine based on the input data which class the instance belong to. One could hope to find some function that gives the answer to this, i.e.
$$\text{class} = f(\text{data})$$
but in real world problems which are complicated enough that it makes sense to apply machine learning, the input data is insufficient to give a definite output. This could be because of the data being to noisy to determine the output with certainty, but it is also possible that even without the noise more than one class is consistent with the input data.

:::{.callout-tip title="Example"}
Assume that we want to train a model to determine the occupation of a person. As input we may use data such as educational background, income, age, gender, etc. This should give the model a good chance to achieve a high accuracy, but since there is not one single track to becoming e.g. a software developer a single, predictive function is impossible to find.
:::

The best we can hope for in such cases is to find a probability distribution function which assigns probabilities to the different possible classes, i.e.
$$p(\text{class}|\text{data},\text{background knowledge}).$$
Here 'data' is the measurement data for the instance we are trying to classify, and 'background knowlege' is the knowledge we have on the relation between different possible measurement data and the true class.

According to Bayes' theorem, we can calculate this as
$$p(\text{class}|\text{data},\text{background knowledge}) 
    = \frac{p(\text{class}|\text{background knowledge})p(\text{data}|\text{class},\text{background knowledge})}{p(\text{data}|\text{background knowledge})}
$$

Here $p(\text{class}|\text{background knowledge})$ is the so-called prior, i.e. the probability we would assign without having the specific information given by the measurement data for that instance. If we know the distribution of the different classes in the population we sample from, we would set the prior equal to that frequency distribution. $p(\text{data}|\text{class},\text{background knowledge})$ is the probability for observing the measurement data we have given the class we are predicting together with the background knowledge. Finally, $p(\text{data}|\text{background knowledge})$ is the probability for observing the measurment data given the background knowledge, but not restricting us to a specific class. 

Unfortunately, this ideal scenario is not often realised. Even if the input distributions are known, or may be well approximated---which is often the case---the computational cost of integrating over all necessary distributions is prohibitive. In practice one need to resort to some approximation. 

### Bayesian Neural Networks (BNN)
BNNs are neural networks in which the weights associated to each node are not single numbers, but probability distributions. In theory, this approach is an example of exact Bayesian inference, but in practice an approximation must be made in the training process. Even with approximations the computational cost of training a BNN is large, making the usability of the approach limited.

Since the weights are replace by probability distributions, the network will produce different outputs if the same instance is fed through it multiple times. The spread of the outputs may be used as and estimate of the epistemic uncertainty of the model.

### Monte Carlo dropout
Monte Carlo dropout is an alternative to BNN with a much lower computational cost. It can be proved to be equivalent to BNN in certain cases,^[[Uncertainty in Deep Learning](https://www.cs.ox.ac.uk/people/yarin.gal/website/thesis/thesis.pdf), Y Gal, PhD thesis, University of Cambridge (2016)] but also for many cases where the proof does not apply, experience indicates that the method yields resonable uncertainty estimates. 

Monte Carlo dropout is a well known technique for regularization of NNs. By randomly removing some of the connections between nodes at each pass during training, single nodes or groups of nodes are less likely to dominate the inference made by the NN reducing the tendency to overtraining. The same technique is now used, but at inference-time. I.e. after training the network, inference is repeated multiple times with Monte Carlo dropout activated. This results in a spread in the output of the NN which can be used to estimate the epistemic uncertainty.

## Deep Ensembles (DE)

An alternative to Bayesian methods is obtained by training the same network with different intialisations. Since there is a randomness in the training process, e.g. through  stochastic decent, each realisation of the trained network will be slightly different. In the most straight-forward application this will only estimate the epistemic uncertainty of the network. It is, however, also possible to estimate the statistical uncertainty by [...]

## Propagation of aleatoric uncertainty

The metods described above are mostly concerned with the uncertainty related to the ML model itself (epistemic), although versions of some of them can also estimate the aleatoric^[In this section the word aleatoric is used to include uncertainty coming from the input data, whether statistical or systematic.] uncertainty, a more straight-forward approach is to estimate the uncertainty of the input data (which is usually done anyway) and then propagate it through the ML model---similar to the approach described in the introduction to this chapter. Different approaches to this propagation has been proposed, for example [Jungmann2024]^[[Analytical Uncertainty Propagation in Neural Networks](https://doi.org/10.1109/TNNLS.2023.3347156), Jungmann et al., IEEE Transactions on Neural Networks and Learning Systems (2024)] which shows that it can be done (at least if correlations are negligible) in a computationally efficient manner which has a close connection to the procedure used for analytical functions.